In [1]:
import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModel

In [2]:
train_df = pd.read_csv("../data/processed_data/train.csv")
dev_df = pd.read_csv("../data/processed_data/dev.csv")

In [3]:
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [4]:
train_texts = train_df["Text_data"].astype(str).tolist()
dev_texts = dev_df["Text_data"].astype(str).tolist()

train_lengths = []

for text in train_texts:
    tokens = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )

    train_lengths.append(len(tokens["input_ids"]))

print("Median token length:", np.median(train_lengths))
print("95th percentile:", np.percentile(train_lengths, 95))
print("Maximum token length:", max(train_lengths))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (522 > 512). Running this sequence through the model will result in indexing errors


Median token length: 125.0
95th percentile: 624.6499999999996
Maximum token length: 4670


In [5]:
print("Texts longer than 256 tokens:", sum(length > 256 for length in train_lengths))
print("Texts longer than 512 tokens:", sum(length > 512 for length in train_lengths))

print("Percentage longer than 256:", np.mean(np.array(train_lengths) > 256) * 100)
print("Percentage longer than 512:", np.mean(np.array(train_lengths) > 512) * 100)

Texts longer than 256 tokens: 1349
Texts longer than 512 tokens: 424
Percentage longer than 256: 24.580903790087465
Percentage longer than 512: 7.725947521865889


In [6]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

Device: mps


In [7]:
model = AutoModel.from_pretrained(model_name)

model = model.to(device)
model.eval()

print("RoBERTa loaded!")
print("Embedding dimensions:", model.config.hidden_size)
print("Maximum supported length:", model.config.max_position_embeddings)

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa loaded!
Embedding dimensions: 768
Maximum supported length: 514


In [17]:
MAX_LENGTH=512

In [18]:
text = train_df["Text_data"].iloc[0]

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = model(**inputs)

print("Original text:")
print(text)

print("\nInput shape:")
print(inputs["input_ids"].shape)

print("\nRoBERTa output shape:")
print(outputs.last_hidden_state.shape)

Original text:
Finally Leaving : I finally left my husband. We still have to cohabitate in the same house for at least a year until we are able to get out from under it and sell. But I just came to the realization that being financially stable didn't make me less angry or more happy when the person I married, who is supposed to say I'm beautiful and smart and funny says all of those things less than a co-worker I have at work. I should have realized those things when we were dating but at that point I was just amazed anyone would want me. I don't like my coworker but he smiles when I walk by, laughs at my jokes, and everyday says "I just wanted you to know you are beautiful." I know it's probably borderline sexual harassment, and he's married as well so it's kind of skeevy but it's depressing that it got to the point of me needing to hear that validation from a stranger, because my husband says those things so infrequently. He gets me treats, and when I get him something in return he a

In [19]:
# Identify special tokens
special_tokens = tokenizer.get_special_tokens_mask(
    inputs["input_ids"][0].tolist(),
    already_has_special_tokens=True
)

special_tokens_mask = torch.tensor(
    special_tokens,
    device=device
).unsqueeze(0)

# Identify actual content tokens
attention_mask = inputs["attention_mask"]

content_mask = attention_mask * (1 - special_tokens_mask)

# Prepare the mask for the 768-dimensional vectors
content_mask = content_mask.unsqueeze(-1).float()

# Get the token representations
token_embeddings = outputs.last_hidden_state

# Add the representations of all content tokens
sum_embeddings = (token_embeddings * content_mask).sum(dim=1)

# Count the content tokens
number_of_tokens = content_mask.sum(dim=1).clamp(min=1)

# Calculate the average representation
document_embedding = sum_embeddings / number_of_tokens

print("Document embedding shape:", document_embedding.shape)


Document embedding shape: torch.Size([1, 768])


In [13]:
# Select five training texts
sample_texts = train_df["Text_data"].iloc[:5].astype(str).tolist()

# Convert texts into model inputs
inputs = tokenizer(
    sample_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH
)

# Move inputs to the selected device
inputs = {key: value.to(device) for key, value in inputs.items()}

# Extract token representations
with torch.no_grad():
    outputs = model(**inputs)

print("Input shape:", inputs["input_ids"].shape)
print("Token representation shape:", outputs.last_hidden_state.shape)

Input shape: torch.Size([5, 475])
Token representation shape: torch.Size([5, 475, 768])


In [14]:
# Identify special tokens in each text
special_tokens_mask = torch.tensor(
    [
        tokenizer.get_special_tokens_mask(
            ids.tolist(),
            already_has_special_tokens=True
        )
        for ids in inputs["input_ids"]
    ],
    device=device
)

# Exclude padding tokens and special tokens
content_mask = inputs["attention_mask"] * (1 - special_tokens_mask)

# Add a dimension so the mask can be applied to 768-dimensional vectors
content_mask = content_mask.unsqueeze(-1).float()

# Get token representations
token_embeddings = outputs.last_hidden_state

# Sum the content-token representations for each document
sum_embeddings = (token_embeddings * content_mask).sum(dim=1)

# Count content tokens in each document
number_of_tokens = content_mask.sum(dim=1).clamp(min=1)

# Calculate one average vector per document
document_embeddings = sum_embeddings / number_of_tokens

print("Document embeddings shape:", document_embeddings.shape)

Document embeddings shape: torch.Size([5, 768])


In [20]:
def get_embeddings(texts, batch_size=8):
    all_embeddings = []

    model.eval()

    # Process texts in batches
    for start in range(0, len(texts), batch_size):

        batch_texts = texts[start:start + batch_size]

        # Tokenize the current batch
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )

        # Move inputs to the selected device
        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        # Extract RoBERTa representations
        with torch.no_grad():
            outputs = model(**inputs)

        # Identify special tokens
        special_tokens_mask = torch.tensor(
            [
                tokenizer.get_special_tokens_mask(
                    ids.tolist(),
                    already_has_special_tokens=True
                )
                for ids in inputs["input_ids"]
            ],
            device=device
        )

        # Exclude padding and special tokens
        content_mask = (
            inputs["attention_mask"] * (1 - special_tokens_mask)
        ).unsqueeze(-1).float()

        # Mean-pool token representations
        token_embeddings = outputs.last_hidden_state

        sum_embeddings = (
            token_embeddings * content_mask
        ).sum(dim=1)

        number_of_tokens = content_mask.sum(dim=1).clamp(min=1)

        document_embeddings = sum_embeddings / number_of_tokens

        # Store embeddings in CPU memory
        all_embeddings.append(
            document_embeddings.cpu().numpy()
        )

    # Combine all batches into one array
    return np.concatenate(all_embeddings, axis=0)

In [21]:
# Extract text from both datasets
train_texts = train_df["Text_data"].astype(str).tolist()
dev_texts = dev_df["Text_data"].astype(str).tolist()

# Extract RoBERTa embeddings
train_embeddings = get_embeddings(train_texts, batch_size=8)
dev_embeddings = get_embeddings(dev_texts, batch_size=8)

# Check the shapes
print("Training embeddings:", train_embeddings.shape)
print("Development embeddings:", dev_embeddings.shape)

Training embeddings: (5488, 768)
Development embeddings: (1373, 768)


In [22]:
from pathlib import Path
import numpy as np

# Create a directory for the embeddings
output_dir = Path("../data/processed_data/embeddings")
output_dir.mkdir(parents=True, exist_ok=True)

# Save the embeddings
np.save(output_dir / "train_embeddings.npy", train_embeddings)
np.save(output_dir / "dev_embeddings.npy", dev_embeddings)

# Save the corresponding labels
np.save(
    output_dir / "train_labels.npy",
    train_df["Label"].to_numpy()
)

np.save(
    output_dir / "dev_labels.npy",
    dev_df["Label"].to_numpy()
)

print("Embeddings and labels saved successfully!")

Embeddings and labels saved successfully!
